In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
cd drive/MyDrive/

/content/drive/MyDrive


In [4]:
train = pd.read_csv('battery_V5.csv')
test = pd.read_csv('test_battery_V5.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (250000, 94)
테스트 데이터 크기: (50000, 93)


In [5]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 90


In [6]:
kf = KFold(n_splits=15, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f"── Fold {fold + 1} ──")
    X_tr = train.loc[tr_idx, feature_cols]
    y_tr = train.loc[tr_idx, TARGET]
    X_val = train.loc[val_idx, feature_cols]
    y_val = train.loc[val_idx, TARGET]

    model = LGBMRegressor(
        device='gpu',
        n_estimators=1000, learning_rate=0.05, max_depth=7,
        num_leaves=63, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbose=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test[feature_cols]) / 5

── Fold 1 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 448.852
[200]	valid_0's l2: 427.695
[300]	valid_0's l2: 416.827
[400]	valid_0's l2: 407.778
[500]	valid_0's l2: 398.205
[600]	valid_0's l2: 391.145
[700]	valid_0's l2: 385.171
[800]	valid_0's l2: 380.343
[900]	valid_0's l2: 376.011
[1000]	valid_0's l2: 372.118
Did not meet early stopping. Best iteration is:
[999]	valid_0's l2: 372.075
── Fold 2 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 535.971
[200]	valid_0's l2: 510.684
[300]	valid_0's l2: 494.121
[400]	valid_0's l2: 484.858
[500]	valid_0's l2: 476.391
[600]	valid_0's l2: 470.212
[700]	valid_0's l2: 465.574
[800]	valid_0's l2: 460.667
[900]	valid_0's l2: 456.154
[1000]	valid_0's l2: 451.908
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l2: 451.908
── Fold 3 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 500.439
[200]	valid_0's l2: 478.28
[300]	va

In [7]:
oof_mae = mean_absolute_error(train[TARGET], oof_preds)
print(f"OOF MAE: {oof_mae:.4f}")

OOF MAE: 9.1835


In [8]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
submission.to_csv('./submission_V9.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
